**Dave**

- MSE
- RMSE
- MAE
- Summary metrics table
- Written conclusion

In [ ]:
!pip install scikit-learn matplotlib 



import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.neural_network import MLPRegressor
# from utils.preprocessing import df_basic_cleaning_and_split, array_standardise_data
from utils import preprocessing
X_train, X_test, y_train, y_test = df_basic_cleaning_and_split()
X_train, X_test, y_train, y_test = array_standardise_data(X_train, X_test, y_train, y_test)

y_train = y_train.flatten()
y_test = y_test.flatten()

ImportError: attempted relative import with no known parent package

In [2]:
print("Training Model 1 for 10,000 iterations will prob take a few mins (last time it took 7 mins)")

def relu(z): return np.maximum(0, z)
def drelu(a): return (a > 0).astype(float)

def forward(X, W1, b1, w2, b2):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    z2 = a1 @ w2 + b2
    return z2.flatten(), {"a1": a1}

def backprop(X, y, W1, b1, w2, b2):
    N = X.shape[0]
    y_hat, cache = forward(X, W1, b1, w2, b2)
    a1 = cache["a1"]
    
    y = y.reshape(-1, 1)
    y_hat = y_hat.reshape(-1, 1)
    
    dL_dyhat = (y_hat - y) / N
    dL_dz2 = dL_dyhat 
    dw2 = a1.T @ dL_dz2
    db2 = np.sum(dL_dz2, axis=0)
    
    dL_da1 = dL_dz2 @ w2.T
    dL_dz1 = dL_da1 * drelu(a1)
    dW1 = X.T @ dL_dz1
    db1 = np.sum(dL_dz1, axis=0)
    
    return {"dW1": dW1, "db1": db1, "dw2": dw2, "db2": db2}
np.random.seed(2)
num_hidden = 64
W1_t = 0.5 * np.random.randn(X_train.shape[1], num_hidden)
b1_t = np.zeros(num_hidden)
w2_t = 0.5 * np.random.randn(num_hidden, 1)
b2_t = np.zeros(1)

lr = 0.001 
iters = 10000 
for t in range(iters):
    grads = backprop(X_train, y_train, W1_t, b1_t, w2_t, b2_t)
    W1_t -= lr * grads["dW1"]
    b1_t -= lr * grads["db1"]
    w2_t -= lr * grads["dw2"]
    b2_t -= lr * grads["db2"]

y_pred_m1, _ = forward(X_test, W1_t, b1_t, w2_t, b2_t)
m1_standard_mse = mean_squared_error(y_test, y_pred_m1)

print("Training Model 2 (Sklearn)")
clf = MLPRegressor(solver='adam', alpha=1e-5, learning_rate='constant',
                   learning_rate_init=0.001, hidden_layer_sizes=(64,), 
                   random_state=1, max_iter=10000)
clf.fit(X_train, y_train)

y_pred_m2 = clf.predict(X_test)
m2_standard_mse = mean_squared_error(y_test, y_pred_m2)

def plot_metric_comparison(metric_name, m1_score, m2_score):
    fig, axs = plt.subplots(1, 1, figsize=(6, 4)) 
    bars = axs.bar(['Model 1\n(Custom NumPy)', 'Model 2\n(Sklearn)'], 
                   [m1_score, m2_score], 
                   color=["green", "blue"], edgecolor='black', width=0.5)
    
    axs.set_title(f'{metric_name}', fontsize=12)
    axs.set_ylabel(f'{metric_name} (Lower is Better)')

    for bar in bars:
        axs.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                 f'{bar.get_height():.4f}', ha='center', va='bottom')
                 
    plt.tight_layout()
    plt.show()

m1_standard_rmse = np.sqrt(m1_standard_mse)
m2_standard_rmse = np.sqrt(m2_standard_mse)

from sklearn.metrics import mean_absolute_error
m1_standard_mae = mean_absolute_error(y_test, y_pred_m1)
m2_standard_mae = mean_absolute_error(y_test, y_pred_m2)
plot_metric_comparison('Mean Squared Error (MSE)', m1_standard_mse, m2_standard_mse)
plot_metric_comparison('Root Mean Squared Error (RMSE)', m1_standard_rmse, m2_standard_rmse)
plot_metric_comparison('Mean Absolute Error (MAE)', m1_standard_mae, m2_standard_mae)

Training Model 1 for 10,000 iterations will prob take a few mins (last time it took 7 mins)


NameError: name 'X_train' is not defined

In [ ]:
import pandas as pd
metric_data = {
    "Metric": [
        "MSE (Mean Squared Error)", 
        "RMSE (Root Mean Squared Error)", 
        "MAE (Mean Absolute Error)"
    ],
    "Model 1 (Numpy)": [
        m1_standard_mse, 
        m1_standard_rmse, 
        m1_standard_mae
    ],
    "Model 2 (Sklearn MLP)": [
        m2_standard_mse, 
        m2_standard_rmse, 
        m2_standard_mae
    ]
}
metrics_table = pd.DataFrame(metric_data)
metrics_table.set_index("Metric", inplace=True)
display(metrics_table)

,Model 1 (Numpy),Model 2 (Sklearn MLP)
Metric,,
MSE (Mean Squared Error),0.883882,0.334372
RMSE (Root Mean Squared Error),0.940150,0.578249
MAE (Mean Absolute Error),0.728763,0.403311


In [ ]:
#sklearn won by a large margin as you can see all the error is lower. 